In [1]:
import pandas as pd
import numpy as np

In [2]:
tmdb_df = pd.read_csv("tmdb_new.csv")
tmdb_df.head()

,id,title,vote_average,vote_count,release_date,revenue,runtime,budget,original_language,popularity,...,production_countries,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,profit_percentage
0,5,Four Rooms,5.861,2687.0,1995-12-09,4257354.0,98.0,4000000.0,en,9.6284,...,United States of America,"Quinn Hellerman, Lawrence Bender, Tim Roth, La...","Allison Anders, Robert Rodriguez, Alexandre Ro...","Phil Parmet, Rodrigo García, Guillermo Navarro...","Allison Anders, Robert Rodriguez, Alexandre Ro...","Lawrence Bender, Alexandre Rockwell, Quentin T...",Combustible Edison,6.7,114082.0,6.433850
1,6,Judgment Night,6.500,349.0,1993-10-15,12136938.0,109.0,21000000.0,en,4.5607,...,United States of America,"Michael Scranton, Hank McGill, Donovan D. Ross...",Stephen Hopkins,Peter Levy,"Jere Cunningham, Lewis Colick","Gene Levy, Lloyd Segan, Marilyn Vance",Alan Silvestri,6.6,19885.0,-42.205057
2,11,Star Wars,8.204,21103.0,1977-05-25,775398007.0,121.0,11000000.0,en,67.2931,...,United States of America,"Paul Blake, Hal Wamsley, David Prowse, Larry W...",George Lucas,Gilbert Taylor,George Lucas,"Gary Kurtz, George Lucas, Rick McCallum",John Williams,8.6,1504993.0,6949.072791
3,12,Finding Nemo,7.815,19616.0,2003-05-30,940335536.0,100.0,94000000.0,en,18.2106,...,United States of America,"Jeff Pidgeon, Danny Mann, Jessie Flower, Bill ...",Andrew Stanton,"Sharon Calahan, Jeremy Lasky","Bob Peterson, Will Csaklos, Ronnie del Carmen,...","John Lasseter, Graham Walters",Thomas Newman,8.2,1158248.0,900.356953
4,13,Forrest Gump,8.468,28113.0,1994-06-23,677387716.0,142.0,55000000.0,en,27.4853,...,United States of America,"Margo Moorer, Joe Stefanelli, Ed Davis, Elizab...",Robert Zemeckis,Don Burgess,"Winston Groom, Eric Roth","Wendy Finerman, Steve Tisch, Steve Starkey",Alan Silvestri,8.8,2374308.0,1131.614029


In [3]:
from collections import Counter

In [4]:
from scipy.sparse import save_npz
from scipy.sparse import csr_matrix


def str_para_categorico(df: pd.DataFrame, coluna: str, k: int) -> pd.DataFrame:
    c = Counter()
    for string in df[coluna]:
        if string is not np.nan:
            c.update(string.split(", ")) 
    res = pd.DataFrame(index=df["title"])
    
    n_cols = 0
    for i in c.most_common():
        n_cols += 1
        if i[1] < k:
            break
    print(n_cols, "colunas")

    for i in c.most_common():
        if i[1] < k:
            break
        nova_col = []
        for string in df[coluna]:
            if string is not np.nan:
                nova_col.append(1 if i[0] in string.split(", ") else 0)
            else:
                nova_col.append(0)
        res[i[0]] = np.array(nova_col, dtype=np.int8)
        res = res.copy()

    sparse = csr_matrix(res) 
    save_npz(f"sparse_{coluna}.npz", sparse) # Salva em forma de matriz esparsa scipy

    return res

In [ ]:
df_cast = str_para_categorico(tmdb_df, "music_composer", k=5)

In [5]:
from scipy.sparse import load_npz
from scipy.sparse import hstack
import os

def abrir_matrizes_esparsas():
    l = []
    for arquivo in os.listdir("./matrizes_esparsas"):
        if arquivo[-4:] == ".npz":
            l.append(load_npz(f"./matrizes_esparsas/{arquivo}"))
    return hstack(l)

In [6]:
sparse = abrir_matrizes_esparsas()

In [7]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(sparse, tmdb_df["revenue"].to_numpy(), test_size=0.1, random_state=42)

In [23]:
from sklearn.linear_model import RidgeCV

l = RidgeCV()
l.fit(X_treino, y_treino)

RidgeCV()

In [20]:
import pickle

#with open("modelos/lasso-imdb_rating.pkl", "wb") as f:
#    pickle.dump(l, f)

In [21]:
#with open("modelos/lasso-imdb_rating.pkl", "rb") as f:
#    l = pickle.load(f)

In [24]:
from sklearn.metrics import root_mean_squared_error

print(root_mean_squared_error(y_treino, l.predict(X_treino)))
root_mean_squared_error(y_teste, l.predict(X_teste))

46829495.19689522


88673240.49427594

In [131]:
tmdb_df["imdb_rating"].describe()

count    12020.000000
mean         6.344651
std          1.137976
min          1.000000
25%          5.700000
50%          6.400000
75%          7.100000
max         10.000000
Name: imdb_rating, dtype: float64